# K-Fall Dataset: CNN-LSTM với kỹ thuật Leave-One-Subject-Out (LOSO) Cross-Validation

### Nguyên lý Leave-One-Subject-Out (LOSO)
Trong bài toán nhận diện hành động người (HAR) và phát hiện té ngã (Fall Detection), đặc trưng vận động của mỗi người (chiều cao, cân nặng, dáng đi, biên độ chuyển động) rất khác nhau.
- **Vấn đề của việc chia ngẫu nhiên**: Dễ dẫn đến rò rỉ dữ liệu (Data Leakage) nếu các cửa sổ của cùng một người xuất hiện ở cả Train và Test.
- **LOSO Cross-Validation**:
  - Với mỗi vòng lặp (Fold), chọn **1 subject** ra làm tập Test độc lập.
  - Toàn bộ các subject còn lại được dùng để Train (và trích 1-2 subject làm Validation cho Early Stopping).
  - Đánh giá mô hình trên subject bị loại ra (người hoàn toàn mới đối với mô hình).
  - Lặp lại lần lượt cho từng subject và tính trung bình hiệu năng tổng thể.

> [!NOTE]
> Để tránh việc phải đọc lại 5.075 file CSV ở mỗi fold, notebook này áp dụng kỹ thuật **Pre-extracting & In-Memory Caching**: toàn bộ cửa sổ của từng subject được tạo 1 lần duy nhất trong RAM, sau đó các fold LOSO ghép mảng tức thì trong < 0.1 giây.


In [ ]:
import os
import re
import glob
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

print("TensorFlow Version:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("Available GPUs:", gpus if gpus else "Running on CPU")


## 1. Đọc và phân tích cấu trúc Label Data (`kfall/label_data_new/`)

Đọc 32 file Excel nhãn của K-Fall bằng `pandas/openpyxl` (kèm auto-fallback an toàn), forward-fill `Task Code` và tạo bảng tra cứu $O(1)$: `(subject, task_id, trial_id) -> (onset_frame, impact_frame)`.


In [ ]:
def parse_kfall_label_file(xlsx_path):
    """Đọc file label bằng pandas (openpyxl), tự động fallback an toàn nếu kernel chưa có openpyxl."""
    try:
        # Cách 1: Ưu tiên dùng pandas + openpyxl
        df = pd.read_excel(xlsx_path)
        task_col = "Task Code (Task ID)"
        if task_col in df.columns:
            df[task_col] = df[task_col].ffill()
        df = df.dropna(subset=[task_col, "Trial ID", "Fall_onset_frame", "Fall_impact_frame"])
        
        sub = Path(xlsx_path).stem.replace("_label", "")
        records = {}
        for _, row in df.iterrows():
            m = re.search(r"\((\d+)\)", str(row[task_col]))
            if m:
                records[(sub, int(m.group(1)), int(row["Trial ID"]))] = (
                    int(row["Fall_onset_frame"]),
                    int(row["Fall_impact_frame"])
                )
        return records
    except Exception:
        # Cách 2: Tự động dự phòng nếu kernel chưa nhận openpyxl
        with zipfile.ZipFile(xlsx_path) as z:
            tree = ET.fromstring(z.read("xl/worksheets/sheet1.xml"))
            rows = []
            for row in tree.findall(".//{*}row"):
                r_dict = {}
                for c in row.findall(".//{*}c"):
                    col = "".join([ch for ch in c.get("r") if ch.isalpha()])
                    t_elem = c.find(".//{*}t")
                    v_elem = c.find(".//{*}v")
                    val = t_elem.text if (t_elem is not None and t_elem.text) else (v_elem.text if (v_elem is not None and v_elem.text) else "")
                    r_dict[col] = val
                rows.append(r_dict)
                
        cur_task = ""
        sub = Path(xlsx_path).stem.replace("_label", "")
        records = {}
        for r in rows[1:]:
            if r.get("A"): cur_task = r["A"].strip()
            trial = r.get("C", "").strip()
            onset = r.get("D", "").strip()
            impact = r.get("E", "").strip()
            if not trial or not onset or not impact: continue
            m = re.search(r"\((\d+)\)", cur_task)
            if m:
                records[(sub, int(m.group(1)), int(float(trial)))] = (
                    int(float(onset)),
                    int(float(impact))
                )
        return records

def load_all_kfall_label_lookup(label_dir):
    """Tải toàn bộ label của 32 subjects thành bảng tra cứu O(1): (sub, task, trial) -> (onset, impact)."""
    files = sorted(glob.glob(os.path.join(label_dir, "*.xlsx")))
    lookup = {}
    for f in files:
        lookup.update(parse_kfall_label_file(f))
    print(f"Đã đọc {len(files)} file nhãn, tổng cộng {len(lookup)} lượt thử ngã (Fall trials).")
    return lookup

# Đường dẫn thư mục dữ liệu
LABEL_DIR = "kfall/label_data_new"
SENSOR_DIR = "kfall/sensor_data_new"

label_lookup = load_all_kfall_label_lookup(LABEL_DIR)


## 2. Thiết lập Hyperparameters & Cấu hình 32 Folds LOSO

- `WINDOW_SIZE = 200`: 2 giây ở 100 Hz.
- `WINDOW_STEP = 100`: Bước nhảy 100 mẫu (overlap 50%).
- `FALL_RATIO_THRESHOLD = 0.20`: Ngưỡng gán nhãn cửa sổ ngã ($\ge 20\%$ frame ngã).
- `DECISION_THRESHOLD = 0.50`: Ngưỡng quyết định cố định 0.50.
- **Đúng chuẩn LOSO trên K-Fall**: Bộ dữ liệu có đúng **32 subjects** (`SA06` - `SA38`, khuyết `SA34`) $\rightarrow$ Chạy **đúng 32 folds** (mỗi fold loại ra 1 người làm Test, 31 người còn lại làm Train/Val).


In [ ]:
WINDOW_SIZE = 200
WINDOW_STEP = 100
FALL_RATIO_THRESHOLD = 0.20
DECISION_THRESHOLD = 0.50

FEATURE_COLS = ["AccX", "AccY", "AccZ"]
# Nếu muốn dùng 6 đặc trưng (Acc + Gyr), mở dòng dưới:
# FEATURE_COLS = ["AccX", "AccY", "AccZ", "GyrX", "GyrY", "GyrZ"]

# Lấy toàn bộ 32 subjects trong thư mục sensor của K-Fall
all_subjects = sorted([
    d for d in os.listdir(SENSOR_DIR)
    if os.path.isdir(os.path.join(SENSOR_DIR, d)) and d.startswith("SA")
])

# Chạy đầy đủ 32 folds tương ứng với 32 subjects
LOSO_TEST_SUBJECTS = all_subjects

print(f"Tổng số subjects có trong K-Fall: {len(all_subjects)}")
print(f"Số folds LOSO sẽ thực hiện: {len(LOSO_TEST_SUBJECTS)} folds")
print(f"Danh sách 32 subjects: {LOSO_TEST_SUBJECTS}")


## 3. Tạo và Caching Sliding Windows cho từng Subject (In-Memory)

Trích xuất toàn bộ sliding windows cho từng subject **một lần duy nhất** và lưu vào từ điển RAM `subject_data[subject_id] = (X_sub, y_sub)`.
Khi chạy vòng lặp LOSO, mỗi fold chỉ việc lấy trực tiếp mảng từ RAM mà không cần đọc lại file từ ổ đĩa.


In [ ]:
FILENAME_PATTERN = re.compile(r"^S(\d{2})T(\d{2})R(\d{2})\.csv$")

def extract_windows_for_single_subject(
    sub,
    sensor_dir,
    label_lookup,
    feature_cols,
    window_size=200,
    step=100,
    fall_ratio_threshold=0.20
):
    """Tạo windows cho 1 subject duy nhất từ các file CSV."""
    windows = []
    labels = []
    use_cols = ["FrameCounter"] + feature_cols
    
    sub_dir = os.path.join(sensor_dir, sub)
    csv_files = sorted(glob.glob(os.path.join(sub_dir, "*.csv")))
    
    for file_path in csv_files:
        fname = os.path.basename(file_path)
        m = FILENAME_PATTERN.match(fname)
        if not m:
            continue
            
        task_id = int(m.group(2))
        trial_id = int(m.group(3))
        
        df = pd.read_csv(file_path, usecols=use_cols)
        feats = df[feature_cols].to_numpy(dtype=np.float32)
        frames = df["FrameCounter"].to_numpy()
        n_samples = len(feats)
        
        if n_samples < window_size:
            continue
            
        if (sub, task_id, trial_id) in label_lookup:
            onset, impact = label_lookup[(sub, task_id, trial_id)]
            fall_mask = (frames >= onset) & (frames <= impact)
        else:
            fall_mask = np.zeros(n_samples, dtype=bool)
            
        for start in range(0, n_samples - window_size + 1, step):
            end = start + window_size
            win_feats = feats[start:end]
            win_mask = fall_mask[start:end]
            fall_ratio = np.mean(win_mask)
            is_fall_win = int(fall_ratio >= fall_ratio_threshold)
            
            windows.append(win_feats)
            labels.append(is_fall_win)
            
    if len(windows) == 0:
        return np.empty((0, window_size, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=np.int8)
        
    return np.asarray(windows, dtype=np.float32), np.asarray(labels, dtype=np.int8)

print("Đang tiền xử lý và nạp dữ liệu cửa sổ cho toàn bộ subjects vào RAM...")
subject_data = {}
for sub in all_subjects:
    X_sub, y_sub = extract_windows_for_single_subject(
        sub, SENSOR_DIR, label_lookup, FEATURE_COLS,
        window_size=WINDOW_SIZE, step=WINDOW_STEP, fall_ratio_threshold=FALL_RATIO_THRESHOLD
    )
    subject_data[sub] = (X_sub, y_sub)
    n_fall = int(np.sum(y_sub == 1))
    n_norm = int(np.sum(y_sub == 0))
    print(f"  {sub}: {len(y_sub):4d} windows (Normal: {n_norm:4d}, Fall: {n_fall:3d})")

print("\nHoàn tất caching toàn bộ subjects vào RAM!")


## 4. Data Augmentation (Kỹ thuật từ `model_3_features`)

Áp dụng 3 kỹ thuật tăng cường dữ liệu cảm biến quán tính IMU **CHỈ TRÊN TẬP TRAIN của mỗi fold**:
- `P_JITTER = 0.10`: Nhiễu Gaussian $\mathcal{N}(0, 0.05)$.
- `P_SCALING = 0.10`: Co giãn biên độ gia tốc $\mathcal{N}(1.0, 0.10)$.
- `P_ROTATION = 0.10`: Xoay 3D vector gia tốc trong khoảng $[-\pi/18, +\pi/18]$ ($\pm 10^\circ$).


In [ ]:
P_JITTER = 0.10
P_SCALING = 0.10
P_ROTATION = 0.10

ACC_AXIS_NAMES = ["AccX", "AccY", "AccZ"]
ACC_AXIS_INDICES = [FEATURE_COLS.index(name) for name in ACC_AXIS_NAMES if name in FEATURE_COLS]

def augment_jitter_window(window, sigma=0.05):
    augmented = window.copy()
    if len(ACC_AXIS_INDICES) == 3:
        noise = np.random.normal(0.0, sigma, size=(WINDOW_SIZE, 3)).astype(np.float32)
        augmented[:, ACC_AXIS_INDICES] += noise
    return augmented

def augment_scaling_window(window, sigma=0.10):
    augmented = window.copy()
    if len(ACC_AXIS_INDICES) == 3:
        acc_scale = np.float32(np.random.normal(1.0, sigma))
        augmented[:, ACC_AXIS_INDICES] *= acc_scale
    return augmented

def augment_rotation_window(window):
    augmented = window.copy()
    angle_range = np.pi / 18
    ax, ay, az = np.random.uniform(-angle_range, angle_range, size=3)

    Rx = np.array([[1.0, 0.0, 0.0], [0.0, np.cos(ax), -np.sin(ax)], [0.0, np.sin(ax), np.cos(ax)]], dtype=np.float32)
    Ry = np.array([[np.cos(ay), 0.0, np.sin(ay)], [0.0, 1.0, 0.0], [-np.sin(ay), 0.0, np.cos(ay)]], dtype=np.float32)
    Rz = np.array([[np.cos(az), -np.sin(az), 0.0], [np.sin(az), np.cos(az), 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
    rotation = Rz @ Ry @ Rx

    if len(ACC_AXIS_INDICES) == 3:
        augmented[:, ACC_AXIS_INDICES] = augmented[:, ACC_AXIS_INDICES] @ rotation.T
    return augmented

def apply_augmentations(X, y):
    augmented_windows = X.copy()
    y_aug = y.copy()
    for i in range(len(X)):
        window = augmented_windows[i].copy()
        if np.random.rand() < P_JITTER:
            window = augment_jitter_window(window)
        if np.random.rand() < P_SCALING:
            window = augment_scaling_window(window)
        if np.random.rand() < P_ROTATION:
            window = augment_rotation_window(window)
        augmented_windows[i] = window
    return augmented_windows, y_aug


## 5. Kiến trúc mô hình CNN-LSTM

Hàm khởi tạo mô hình mới cho từng Fold:
- 2 tầng Conv1D (BatchNormalization, ReLU, MaxPooling1D, Dropout).
- 1 tầng LSTM (64 units).
- Tầng phân loại Dense Sigmoid.


In [ ]:
def build_cnn_lstm_model(input_shape=(200, 3)):
    inputs = layers.Input(shape=input_shape, name="imu_input")
    
    # Block 1: Conv1D
    x = layers.Conv1D(filters=32, kernel_size=5, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    
    # Block 2: Conv1D
    x = layers.Conv1D(filters=64, kernel_size=3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)
    x = layers.Dropout(0.2)(x)
    
    # Block 3: LSTM
    x = layers.LSTM(64, return_sequences=False, dropout=0.2, recurrent_dropout=0.0)(x)
    
    # Dense Classifier
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="fall_probability")(x)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="CNN_LSTM_FallDetector")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model


## 6. Vòng lặp huấn luyện & đánh giá Leave-One-Subject-Out (LOSO)

Ở mỗi Fold $k$:
1. **Test Set**: Subject $k$.
2. **Validation Set**: 1 subject từ danh sách còn lại (dùng để Early Stopping).
3. **Train Set**: Toàn bộ các subject còn lại.
4. **Data Augmentation**: Nhân đôi mẫu trên tập Train (`X_train_combined`).
5. **StandardScaler**: Fit **chỉ trên Train gốc**, sau đó transform cho Train mở rộng, Val và Test.
6. **Huấn luyện**: EarlyStopping dựa trên `val_recall` (`patience=10`, `restore_best_weights=True`).
7. **Đánh giá**: Dự đoán trên Test Subject $k$ tại ngưỡng cố định `DECISION_THRESHOLD = 0.50`.


In [ ]:
os.makedirs("saved_models_cnn_lstm", exist_ok=True)
num_features = len(FEATURE_COLS)

loso_results = []
all_loso_y_true = []
all_loso_y_pred = []
all_loso_y_prob = []
all_loso_subjects = []

print("=" * 80)
print(f" BẮT ĐẦU CHẠY ĐẦY ĐỦ LOSO CROSS-VALIDATION VỚI {len(LOSO_TEST_SUBJECTS)} FOLDS (32 SUBJECTS)")
print("=" * 80)

for fold_idx, test_sub in enumerate(LOSO_TEST_SUBJECTS, start=1):
    print(f"\n>>> [FOLD {fold_idx:02d}/{len(LOSO_TEST_SUBJECTS):02d}] TEST SUBJECT: {test_sub}")
    
    # 1. Dữ liệu Test của subject này
    X_test, y_test = subject_data[test_sub]
    
    # 2. 31 subjects còn lại
    remaining_subs = [s for s in all_subjects if s != test_sub]
    
    # Chọn xoay vòng 2 subjects làm Validation cho EarlyStopping, 29 subjects còn lại làm Train
    val_sub_1 = remaining_subs[(fold_idx - 1) % len(remaining_subs)]
    val_sub_2 = remaining_subs[fold_idx % len(remaining_subs)]
    val_subs = [val_sub_1, val_sub_2]
    train_subs = [s for s in remaining_subs if s not in val_subs]
    
    print(f"    - Train: {len(train_subs)} subjects | Val: {val_subs}")
    print(f"    - Test : {test_sub} ({len(y_test)} windows: Normal={np.sum(y_test==0)}, Fall={np.sum(y_test==1)})")
    
    # Ghép dữ liệu Train và Val từ RAM (cực nhanh, không đọc lại đĩa)
    X_train_raw = np.concatenate([subject_data[s][0] for s in train_subs], axis=0)
    y_train = np.concatenate([subject_data[s][1] for s in train_subs], axis=0)
    
    X_val_raw = np.concatenate([subject_data[s][0] for s in val_subs], axis=0)
    y_val = np.concatenate([subject_data[s][1] for s in val_subs], axis=0)
    
    # 3. Augment CHỈ TRÊN TẬP TRAIN
    X_train_aug_raw, y_train_aug = apply_augmentations(X_train_raw, y_train)
    X_train_comb_raw = np.concatenate([X_train_raw, X_train_aug_raw], axis=0)
    y_train_comb = np.concatenate([y_train, y_train_aug], axis=0)
    
    # 4. Fit StandardScaler CHỈ trên X_train_raw
    scaler = StandardScaler()
    scaler.fit(X_train_raw.reshape(-1, num_features))
    
    X_train_scaled = scaler.transform(X_train_comb_raw.reshape(-1, num_features)).reshape(-1, WINDOW_SIZE, num_features)
    X_val_scaled = scaler.transform(X_val_raw.reshape(-1, num_features)).reshape(-1, WINDOW_SIZE, num_features)
    X_test_scaled = scaler.transform(X_test.reshape(-1, num_features)).reshape(-1, WINDOW_SIZE, num_features)
    
    # 5. Class Weights
    total_samples = len(y_train_comb)
    neg_count = np.sum(y_train_comb == 0)
    pos_count = np.sum(y_train_comb == 1)
    class_weight = {
        0: (total_samples / (2.0 * neg_count)),
        1: (total_samples / (2.0 * pos_count))
    }
    
    # 6. Khởi tạo và Huấn luyện Model mới cho Fold này
    fold_model_path = f"saved_models_cnn_lstm/best_model_{test_sub}.keras"
    fold_model = build_cnn_lstm_model(input_shape=(WINDOW_SIZE, num_features))
    
    cb_list = [
        callbacks.ModelCheckpoint(
            filepath=fold_model_path,
            monitor="val_recall",
            mode="max",
            save_best_only=True,
            save_weights_only=False,
            verbose=0
        ),
        callbacks.EarlyStopping(
            monitor="val_recall",
            mode="max",
            patience=12,
            min_delta=1e-3,
            restore_best_weights=True,
            verbose=0
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=3,
            min_lr=1e-5,
            verbose=0
        )
    ]
    
    fold_model.fit(
        X_train_scaled, y_train_comb,
        validation_data=(X_val_scaled, y_val),
        batch_size=32,
        epochs=50,
        class_weight=class_weight,
        callbacks=cb_list,
        verbose=0
    )
    
    # Nạp lại checkpoint tốt nhất của fold
    try:
        eval_model = tf.keras.models.load_model(fold_model_path, compile=False)
    except Exception:
        eval_model = fold_model
        
    # 7. Dự đoán và tính metrics trên Test Subject với threshold = 0.50
    test_probs = eval_model.predict(X_test_scaled, verbose=0).ravel()
    test_preds = (test_probs >= DECISION_THRESHOLD).astype(int)
    
    acc = accuracy_score(y_test, test_preds)
    prec = precision_score(y_test, test_preds, pos_label=1, zero_division=0)
    rec = recall_score(y_test, test_preds, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, test_preds, pos_label=1, zero_division=0)
    f2 = fbeta_score(y_test, test_preds, beta=2, zero_division=0)
    
    if len(np.unique(y_test)) == 2:
        auc = roc_auc_score(y_test, test_probs)
    else:
        auc = np.nan
        
    tn, fp, fn, tp = confusion_matrix(y_test, test_preds, labels=[0, 1]).ravel()
    
    print(f"    -> Fold {test_sub}: Acc={acc:.4f} | Recall={rec:.4f} | Precision={prec:.4f} | F1={f1:.4f} | F2={f2:.4f} (TP={tp}, FN={fn}, FP={fp}, TN={tn})")
    
    loso_results.append({
        "Fold": fold_idx,
        "Test_Subject": test_sub,
        "Windows": len(y_test),
        "Normal": int(np.sum(y_test == 0)),
        "Fall": int(np.sum(y_test == 1)),
        "Accuracy": acc,
        "Precision_Fall": prec,
        "Recall_Fall": rec,
        "F1_Fall": f1,
        "F2_Fall": f2,
        "ROC_AUC": auc,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    })
    
    all_loso_y_true.append(y_test)
    all_loso_y_pred.append(test_preds)
    all_loso_y_prob.append(test_probs)
    all_loso_subjects.append(np.full(len(y_test), test_sub))

print("\n" + "=" * 80)
print(" ĐÃ HOÀN TẤT TOÀN BỘ 32 FOLDS CỦA LOSO!")
print("=" * 80)


## 7. Tổng hợp và Phân tích Kết quả LOSO Cross-Validation

Bảng tổng hợp chi tiết từng fold, giá trị Trung bình (Mean) và Độ lệch chuẩn (Std) của mô hình trên người dùng mới.


In [ ]:
loso_df = pd.DataFrame(loso_results)

print("=" * 95)
print(" BẢNG KẾT QUẢ TỪNG FOLD LEAVE-ONE-SUBJECT-OUT (LOSO)")
print("=" * 95)
display(loso_df.round(4))

# Tính Mean và Std qua các folds
metrics_cols = ["Accuracy", "Precision_Fall", "Recall_Fall", "F1_Fall", "F2_Fall", "ROC_AUC"]
mean_metrics = loso_df[metrics_cols].mean()
std_metrics = loso_df[metrics_cols].std()

summary_table = pd.DataFrame({
    "Mean": mean_metrics,
    "Std": std_metrics,
    "Min": loso_df[metrics_cols].min(),
    "Max": loso_df[metrics_cols].max()
})

print("\n" + "=" * 65)
print(" THỐNG KÊ TỔNG HỢP CÁC FOLD (MEAN ± STD)")
print("=" * 65)
display(summary_table.round(4))

# Gộp toàn bộ dự đoán các folds để xem tổng thể
pooled_y_true = np.concatenate(all_loso_y_true)
pooled_y_pred = np.concatenate(all_loso_y_pred)
pooled_y_prob = np.concatenate(all_loso_y_prob)

total_tn, total_fp, total_fn, total_tp = confusion_matrix(pooled_y_true, pooled_y_pred, labels=[0, 1]).ravel()

print("\n" + "=" * 65)
print(" MA TRẬN NHẦM LẪN TỔNG HỢP TOÀN BỘ DỮ LIỆU LOSO")
print("=" * 65)
print(f"Tổng số windows đánh giá: {len(pooled_y_true)}")
print(f"TP (Phát hiện đúng ngã):     {total_tp:4d}")
print(f"FN (Bỏ sót ngã):             {total_fn:4d}")
print(f"FP (Báo động giả):           {total_fp:4d}")
print(f"TN (Bình thường đúng):       {total_tn:4d}")

print("\nClassification Report Tổng hợp:")
print(classification_report(
    pooled_y_true, pooled_y_pred,
    labels=[0, 1],
    target_names=["Bình thường (0)", "Ngã (1)"],
    digits=4,
    zero_division=0
))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Confusion Matrix tổng
cm = confusion_matrix(pooled_y_true, pooled_y_pred, labels=[0, 1])
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Normal", "Fall"]).plot(
    cmap="Blues", values_format="d", ax=axes[0]
)
axes[0].set_title("Pooled Confusion Matrix - ALL LOSO Folds")

# 2. Biểu đồ Recall & Precision qua từng fold
x_labels = loso_df["Test_Subject"].tolist()
x_indices = np.arange(len(x_labels))
width = 0.35

axes[1].bar(x_indices - width/2, loso_df["Recall_Fall"], width, label="Recall Fall", color="#2ca02c", alpha=0.85)
axes[1].bar(x_indices + width/2, loso_df["Precision_Fall"], width, label="Precision Fall", color="#1f77b4", alpha=0.85)
axes[1].set_xticks(x_indices)
axes[1].set_xticklabels(x_labels, rotation=45)
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Recall & Precision qua từng Fold")
axes[1].set_ylabel("Score")
axes[1].axhline(0.90, color="red", linestyle="--", alpha=0.6, label="Ngưỡng 90%")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Train model cuối cùng trên toàn bộ 32 subject

Chỉ chạy cell này **sau khi đã chốt cấu hình bằng LOSO**. Cell có hai pha: (1) dùng validation nội bộ chỉ để chọn epoch bằng callback như pipeline train thường; (2) khởi tạo model mới và train trên 100% window của cả 32 subject theo epoch đã chọn. Model lưu để deploy là model pha (2).


In [ ]:
# Train model deploy trên toàn bộ 32 subject — không dùng kết quả này để báo cáo metric.
# Pha chọn epoch dùng callback cùng cấu hình train thông thường.
FINAL_MAX_EPOCHS = 100
FINAL_VAL_FRACTION = 0.10
FINAL_USE_AUGMENTATION = True  # Phải khớp pipeline LOSO đã dùng để đánh giá.
FINAL_OUTPUT_DIR = "saved_models_cnn_lstm"
FINAL_SEED = 42

if len(all_subjects) != 32:
    raise ValueError(f"Expected 32 subjects, found {len(all_subjects)}: {all_subjects}")

tf.keras.utils.set_random_seed(FINAL_SEED)
num_features = len(FEATURE_COLS)
X_all_raw = np.concatenate([subject_data[s][0] for s in all_subjects], axis=0)
y_all = np.concatenate([subject_data[s][1] for s in all_subjects], axis=0)

# Scaler deploy được fit trên toàn bộ dữ liệu train cuối cùng.
final_scaler = StandardScaler()
final_scaler.fit(X_all_raw.reshape(-1, num_features))

if FINAL_USE_AUGMENTATION:
    X_aug_raw, y_aug = apply_augmentations(X_all_raw, y_all)
    X_final_raw = np.concatenate([X_all_raw, X_aug_raw], axis=0)
    y_final = np.concatenate([y_all, y_aug], axis=0)
else:
    X_final_raw = X_all_raw
    y_final = y_all

X_final = final_scaler.transform(X_final_raw.reshape(-1, num_features)).reshape(
    -1, WINDOW_SIZE, num_features
)

total_samples = len(y_final)
neg_count = int(np.sum(y_final == 0))
pos_count = int(np.sum(y_final == 1))
if neg_count == 0 or pos_count == 0:
    raise ValueError("Both Normal and Fall classes are required to train the final model.")
class_weight = {
    0: total_samples / (2.0 * neg_count),
    1: total_samples / (2.0 * pos_count),
}

print(f"Training on {len(all_subjects)} subjects / {len(y_final)} windows (augmentation={FINAL_USE_AUGMENTATION})")
print(f"Normal={neg_count}, Fall={pos_count}, class_weight={class_weight}")

os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)
final_model_path = os.path.join(FINAL_OUTPUT_DIR, "final_model.keras")
selection_model_path = os.path.join(FINAL_OUTPUT_DIR, "final_selection_best.keras")
final_scaler_path = os.path.join(FINAL_OUTPUT_DIR, "scaler.pkl")
final_metadata_path = os.path.join(FINAL_OUTPUT_DIR, "metadata.pkl")

# Xáo trộn trước validation_split để cả train/validation đều có window từ nhiều subject.
rng = np.random.default_rng(FINAL_SEED)
permutation = rng.permutation(len(y_final))
X_selection = X_final[permutation]
y_selection = y_final[permutation]

selection_model = build_cnn_lstm_model(input_shape=(WINDOW_SIZE, num_features))
selection_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=selection_model_path, monitor="val_recall", mode="max",
        save_best_only=True, save_weights_only=False, verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_recall", mode="max", patience=12, min_delta=1e-3,
        restore_best_weights=True, verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", mode="min", factor=0.5, patience=3,
        min_lr=1e-5, verbose=1,
    ),
]
selection_history = selection_model.fit(
    X_selection, y_selection, validation_split=FINAL_VAL_FRACTION,
    batch_size=32, epochs=FINAL_MAX_EPOCHS, class_weight=class_weight,
    callbacks=selection_callbacks, verbose=1,
)
selected_epoch = int(np.argmax(selection_history.history["val_recall"]) + 1)
selected_val_recall = float(np.max(selection_history.history["val_recall"]))
print(f"Selected epoch from internal val_recall: {selected_epoch} ({selected_val_recall:.6f})")

# Model deploy: khởi tạo mới và dùng tất cả window của đủ 32 subject.
tf.keras.utils.set_random_seed(FINAL_SEED)
final_model = build_cnn_lstm_model(input_shape=(WINDOW_SIZE, num_features))
final_history = final_model.fit(
    X_final, y_final, batch_size=32, epochs=selected_epoch,
    class_weight=class_weight, verbose=1,
)
final_model.save(final_model_path)
import joblib
joblib.dump(final_scaler, final_scaler_path)
joblib.dump({
    "feature_cols": FEATURE_COLS,
    "window_size": WINDOW_SIZE,
    "window_step": WINDOW_STEP,
    "fall_ratio_threshold": FALL_RATIO_THRESHOLD,
    "decision_threshold": DECISION_THRESHOLD,
    "subjects": all_subjects,
    "num_subjects": len(all_subjects),
    "augmentation": FINAL_USE_AUGMENTATION,
    "selection_max_epochs": FINAL_MAX_EPOCHS,
    "selection_val_fraction": FINAL_VAL_FRACTION,
    "selected_epoch": selected_epoch,
    "selected_val_recall": selected_val_recall,
    "seed": FINAL_SEED,
}, final_metadata_path)

print(f"Saved final model: {final_model_path}")
print(f"Saved scaler:      {final_scaler_path}")
print(f"Saved metadata:    {final_metadata_path}")
